# CrashCase: reproduce the damage model on a GPU

Runs the repo's DVC pipeline (`validate -> leakage -> prepare -> baseline -> train`) on a Kaggle GPU with the CarDD dataset attached (`issamjebnouni/cardd`). Needs a GPU and internet turned on.

Outputs in `/kaggle/working`: `damage-seg.pt`, `metrics.json`, `dvc.lock`, the data reports, and the MLflow store (`mlflow.db` + `mlruns.zip`) holding the baseline runs, the YOLO run and the registered model. Copy `dvc.lock`, `metrics.json` and `reports/` back into the repo and commit them; put `mlflow.db` and `mlruns/` in the repo root to browse with `make mlflow-ui`.

In [ ]:
# Branch of the repo to train from
REF = "fix/local-setup"
# DagsHub repo that hosts the DVC remote and MLflow. Add your DagsHub token as the Kaggle
# secret DAGSHUB_TOKEN (Add-ons -> Secrets); without it, runs go to a local mlflow.db.
DAGSHUB_USER = "uzairahm290"
DAGSHUB_REPO = "CrashCase-AI"

!nvidia-smi -L
!git clone --depth 1 --branch {REF} https://github.com/uzairahm290/CrashCase-AI /tmp/CrashCase-AI
# sqlalchemy<2.1: mlflow 2.x imports a pool class that SQLAlchemy 2.1 removed
!pip install -q "ultralytics>=8.2" "mlflow>=2.12,<3" "sqlalchemy<2.1" "scikit-learn>=1.4" "dvc>=3.50"

Lay the Kaggle copy out as `datasets/CarDD_COCO`, the layout the pipeline expects. Copied rather than symlinked so DVC hashes real files.

In [ ]:
import glob
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/tmp/CrashCase-AI")
src = Path(glob.glob("/kaggle/input/**/train.json", recursive=True)[0]).parent
coco = Path("datasets/CarDD_COCO")
(coco / "annotations").mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    shutil.copytree(src / split, coco / f"{split}2017", dirs_exist_ok=True)
    shutil.copy2(src / f"{split}.json", coco / "annotations" / f"instances_{split}2017.json")

The whole pipeline, forced so every stage runs and logs. A failing data check stops it here, before training.

In [ ]:
os.environ["WANDB_MODE"] = "disabled"
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
    os.environ.update(
        MLFLOW_TRACKING_URI=f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow",
        MLFLOW_TRACKING_USERNAME=DAGSHUB_USER,
        MLFLOW_TRACKING_PASSWORD=token,
    )
    print("logging to DagsHub MLflow")
except Exception:
    os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////tmp/CrashCase-AI/mlflow.db"
    print("no DAGSHUB_TOKEN secret: logging to a local mlflow.db")
subprocess.run(["git", "config", "--global", "user.email", "kaggle@example.com"], check=True)
subprocess.run(["git", "config", "--global", "user.name", "kaggle"], check=True)
# check=True stops the notebook on failure; a plain !dvc would carry on
subprocess.run(["dvc", "repro", "--force"], check=True)
subprocess.run(["dvc", "metrics", "show"], check=True)

Keep what the repo and the submission need.

In [ ]:
import json

out = Path("/kaggle/working")
for f in ("models/damage-seg.pt", "metrics.json", "dvc.lock", "mlflow.db"):
    if Path(f).exists():
        shutil.copy2(f, out)
shutil.copytree("reports", out / "reports", dirs_exist_ok=True)
(out / "plots").mkdir(exist_ok=True)
for plot in glob.glob("runs/**/damage-seg/*.png", recursive=True):
    shutil.copy2(plot, out / "plots")
if Path("mlruns").exists():
    shutil.make_archive(str(out / "mlruns"), "zip", "mlruns")
subprocess.run(["python", "scripts/export_mlflow.py", "--out", str(out / "mlflow-export")], check=True)
print((out / "mlflow-export" / "comparison.md").read_text())
print(json.dumps(json.loads(Path("metrics.json").read_text()), indent=2))